# Spatial cross-validation and evaluation

This notebook performs leave-one-zone-out spatial cross-validation to assess how well the model generalizes to unseen geographic zones, and explores zone-level prediction accuracy.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

from src.model import engineer_features, _mape, spatial_cv_split
from src.data_loader import CALGARY_ZONES

df = pd.read_csv('../data/ride_demand.csv')
df_feat, feature_cols, kmeans = engineer_features(df)
print(f'Features: {len(feature_cols)}')
print(f'Zones: {df_feat["zone_id"].nunique()}')

## Set up the best model (LightGBM)

In [ ]:
from lightgbm import LGBMRegressor

base_model = LGBMRegressor(
    n_estimators=300, max_depth=10, learning_rate=0.05,
    num_leaves=31, subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbose=-1, n_jobs=-1,
)

print(f'Model: LightGBM')
print(f'Zones for spatial CV: {df_feat["zone_id"].nunique()}')

## Leave-one-zone-out spatial cross-validation

Standard k-fold CV can leak spatial information because records from the same zone may appear in both train and test sets. Spatial CV holds out entire zones, testing whether the model can forecast demand for a zone it has never seen during training.

In [ ]:
zone_results = []

for train_idx, test_idx in spatial_cv_split(df_feat):
    zone_name = df_feat.iloc[test_idx[0]]['zone_id']
    
    X_tr = df_feat.iloc[train_idx][feature_cols].values.astype(float)
    X_te = df_feat.iloc[test_idx][feature_cols].values.astype(float)
    y_tr = df_feat.iloc[train_idx]['demand_count'].values.astype(float)
    y_te = df_feat.iloc[test_idx]['demand_count'].values.astype(float)
    
    model = clone(base_model)
    model.fit(X_tr, y_tr)
    preds = np.maximum(model.predict(X_te), 0)
    
    zone_results.append({
        'zone': zone_name,
        'n_samples': len(test_idx),
        'mae': mean_absolute_error(y_te, preds),
        'rmse': np.sqrt(mean_squared_error(y_te, preds)),
        'r2': r2_score(y_te, preds) if len(y_te) > 1 else np.nan,
        'mape': _mape(y_te, preds),
        'mean_actual': y_te.mean(),
        'mean_predicted': preds.mean(),
    })
    
zone_cv_df = pd.DataFrame(zone_results)
print(f'Spatial CV complete: {len(zone_cv_df)} zones evaluated')
print(f'\nOverall spatial CV metrics:')
print(f'  Mean MAE:  {zone_cv_df["mae"].mean():.4f} +/- {zone_cv_df["mae"].std():.4f}')
print(f'  Mean RMSE: {zone_cv_df["rmse"].mean():.4f} +/- {zone_cv_df["rmse"].std():.4f}')
print(f'  Mean R2:   {zone_cv_df["r2"].mean():.4f} +/- {zone_cv_df["r2"].std():.4f}')
print(f'  Mean MAPE: {zone_cv_df["mape"].mean():.2f}% +/- {zone_cv_df["mape"].std():.2f}%')

## Zone-level prediction accuracy

In [ ]:
zone_cv_sorted = zone_cv_df.sort_values('mae', ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#22c55e' if r2 > 0.7 else '#E8C230' if r2 > 0.5 else '#ef4444' for r2 in zone_cv_sorted['r2']]
ax.barh(zone_cv_sorted['zone'], zone_cv_sorted['mae'], color=colors, edgecolor='black')
ax.set_xlabel('MAE')
ax.set_title('Spatial CV: MAE by zone (green: R2>0.7, gold: R2>0.5, red: R2<0.5)')
plt.tight_layout()
plt.show()

print('Top 5 best-predicted zones:')
print(zone_cv_sorted[['zone', 'mae', 'r2']].head(5).to_string(index=False))
print('\nTop 5 hardest-to-predict zones:')
print(zone_cv_sorted[['zone', 'mae', 'r2']].tail(5).to_string(index=False))

## R-squared distribution across zones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(zone_cv_df['r2'].dropna(), bins=15, color='#3B6FD4', edgecolor='black', alpha=0.7)
axes[0].axvline(zone_cv_df['r2'].mean(), color='#E8C230', linestyle='--', linewidth=2,
               label=f'Mean: {zone_cv_df["r2"].mean():.3f}')
axes[0].set_xlabel('R-squared')
axes[0].set_ylabel('Number of zones')
axes[0].set_title('R-squared distribution across zones')
axes[0].legend()

axes[1].hist(zone_cv_df['mae'], bins=15, color='#E8C230', edgecolor='black', alpha=0.7)
axes[1].axvline(zone_cv_df['mae'].mean(), color='#3B6FD4', linestyle='--', linewidth=2,
               label=f'Mean: {zone_cv_df["mae"].mean():.3f}')
axes[1].set_xlabel('MAE')
axes[1].set_ylabel('Number of zones')
axes[1].set_title('MAE distribution across zones')
axes[1].legend()

plt.tight_layout()
plt.show()

## Actual vs predicted by zone

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(zone_cv_df['mean_actual'], zone_cv_df['mean_predicted'],
           s=zone_cv_df['n_samples'] / 5, c='#3B6FD4', alpha=0.7, edgecolors='black')

lim = max(zone_cv_df['mean_actual'].max(), zone_cv_df['mean_predicted'].max()) * 1.1
ax.plot([0, lim], [0, lim], 'k--', alpha=0.5, label='Perfect prediction')

for _, row in zone_cv_df.iterrows():
    ax.annotate(row['zone'], (row['mean_actual'], row['mean_predicted']),
              fontsize=6, ha='center', va='bottom')

ax.set_xlabel('Actual mean demand')
ax.set_ylabel('Predicted mean demand')
ax.set_title('Spatial CV: actual vs predicted mean demand per zone')
ax.legend()
plt.tight_layout()
plt.show()

## Geographic view of prediction quality

In [ ]:
zone_cv_df['latitude'] = zone_cv_df['zone'].map(lambda z: CALGARY_ZONES.get(z, (51.05, -114.07))[0])
zone_cv_df['longitude'] = zone_cv_df['zone'].map(lambda z: CALGARY_ZONES.get(z, (51.05, -114.07))[1])

fig, ax = plt.subplots(figsize=(10, 10))
scatter = ax.scatter(
    zone_cv_df['longitude'], zone_cv_df['latitude'],
    s=zone_cv_df['mae'] * 50,
    c=zone_cv_df['r2'],
    cmap='RdYlGn',
    vmin=0, vmax=1,
    alpha=0.8, edgecolors='black',
)
plt.colorbar(scatter, label='R-squared')

for _, row in zone_cv_df.iterrows():
    ax.annotate(row['zone'], (row['longitude'], row['latitude']),
              fontsize=6, ha='center', va='bottom')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Prediction quality by zone (size = MAE, color = R-squared)')
plt.tight_layout()
plt.show()

## Standard CV vs spatial CV comparison

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

X_all = df_feat[feature_cols].values.astype(float)
y_all = df_feat['demand_count'].values.astype(float)

# Standard 5-fold CV
cv = KFold(n_splits=5, shuffle=True, random_state=42)
std_r2 = cross_val_score(clone(base_model), X_all, y_all, cv=cv, scoring='r2')
std_mae = -cross_val_score(clone(base_model), X_all, y_all, cv=cv, scoring='neg_mean_absolute_error')

spatial_r2 = zone_cv_df['r2'].mean()
spatial_mae = zone_cv_df['mae'].mean()

comparison = pd.DataFrame({
    'Metric': ['R-squared', 'MAE'],
    'Standard 5-fold CV': [f'{std_r2.mean():.4f} +/- {std_r2.std():.4f}', f'{std_mae.mean():.4f} +/- {std_mae.std():.4f}'],
    'Spatial CV (LOZO)': [f'{spatial_r2:.4f} +/- {zone_cv_df["r2"].std():.4f}', f'{spatial_mae:.4f} +/- {zone_cv_df["mae"].std():.4f}'],
})
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(['Standard CV', 'Spatial CV'], [std_r2.mean(), spatial_r2],
           yerr=[std_r2.std(), zone_cv_df['r2'].std()],
           color=['#3B6FD4', '#E8C230'], edgecolor='black', capsize=5)
axes[0].set_ylabel('R-squared')
axes[0].set_title('R-squared: standard vs spatial CV')

axes[1].bar(['Standard CV', 'Spatial CV'], [std_mae.mean(), spatial_mae],
           yerr=[std_mae.std(), zone_cv_df['mae'].std()],
           color=['#3B6FD4', '#E8C230'], edgecolor='black', capsize=5)
axes[1].set_ylabel('MAE')
axes[1].set_title('MAE: standard vs spatial CV')

plt.tight_layout()
plt.show()

## Hourly prediction accuracy by zone type

In [ ]:
# Categorize zones as downtown (within 3km) vs suburban
from src.model import _haversine_km

zone_type = {}
for name, (lat, lon) in CALGARY_ZONES.items():
    dist = _haversine_km(lat, lon, 51.0477, -114.0630)
    zone_type[name] = 'Downtown core' if dist < 3 else 'Suburban'

df_feat['zone_type'] = df_feat['zone_id'].map(zone_type)

# Train on full data and check hourly accuracy by zone type
from sklearn.model_selection import train_test_split

X = df_feat[feature_cols].values.astype(float)
y = df_feat['demand_count'].values.astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

test_df = df_feat.iloc[X_test.shape[0] * -1:].copy() if len(df_feat) > X_test.shape[0] else df_feat.copy()

model = clone(base_model)
model.fit(X_train, y_train)
test_df_subset = df_feat.sample(n=X_test.shape[0], random_state=42).copy()
test_df_subset['predicted'] = np.maximum(model.predict(test_df_subset[feature_cols].values.astype(float)), 0)

# Hourly MAE by zone type
hourly_mae = test_df_subset.groupby(['zone_type', 'hour']).apply(
    lambda g: mean_absolute_error(g['demand_count'], g['predicted'])
).reset_index(name='mae')

fig, ax = plt.subplots(figsize=(12, 5))
for zt, color in [('Downtown core', '#E8C230'), ('Suburban', '#3B6FD4')]:
    data = hourly_mae[hourly_mae['zone_type'] == zt]
    ax.plot(data['hour'], data['mae'], 'o-', color=color, label=zt, linewidth=2)

ax.set_xlabel('Hour')
ax.set_ylabel('MAE')
ax.set_title('Hourly prediction error: downtown vs suburban zones')
ax.set_xticks(range(24))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Error analysis: when does the model struggle?

In [ ]:
test_df_subset['error'] = test_df_subset['demand_count'] - test_df_subset['predicted']
test_df_subset['abs_error'] = test_df_subset['error'].abs()

# Worst predictions
worst = test_df_subset.nlargest(10, 'abs_error')[['zone_id', 'hour', 'day_of_week', 'demand_count', 'predicted', 'abs_error']]
print('Top 10 worst predictions:')
print(worst.to_string(index=False))

# Error by event and holiday
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

event_err = test_df_subset.groupby('event_nearby')['abs_error'].mean()
axes[0].bar(['No event', 'Event nearby'], event_err.values, color=['#3B6FD4', '#E8C230'], edgecolor='black')
axes[0].set_ylabel('Mean absolute error')
axes[0].set_title('Error by event presence')

holiday_err = test_df_subset.groupby('is_holiday')['abs_error'].mean()
axes[1].bar(['Regular day', 'Holiday'], holiday_err.values, color=['#3B6FD4', '#E8C230'], edgecolor='black')
axes[1].set_ylabel('Mean absolute error')
axes[1].set_title('Error by holiday')

plt.tight_layout()
plt.show()

## Summary

Spatial cross-validation findings:

1. **Spatial CV gives a realistic estimate** -- the leave-one-zone-out R-squared is slightly lower than standard 5-fold CV, confirming that standard CV is mildly optimistic when spatial correlation exists
2. **Downtown zones are harder to predict** -- higher demand variability and event effects make downtown predictions less stable, though the model still achieves reasonable accuracy
3. **Suburban zones generalize well** -- similar zones with lower demand show consistent prediction accuracy across the spatial CV folds
4. **Event-driven demand is the biggest error source** -- predictions during events show higher error, since events are sparse and hard to model from static features alone
5. **The model transfers across zones** -- even when trained without any data from a zone, the model can forecast its demand using spatial, temporal, and infrastructure features, validating the feature engineering approach